In [1]:
import pandas as pd
import re

BENCHMARK_COL = "benchmark"   # 例如 "业绩比较基准" / "基准" 等
index_map = {
    "沪深300": "000300.SH",
    "恒生指数": "HSI.HI",
    "中证港股通": "930930.CSI",
    "中证全债": "H11001.CSI",
    "中证综合债": "H11009.CSI",
    "中证内地消费主题": "000932.SH",    # 爬到了

    "中债新综合财富（总值）": "CBA00101.CS",
    
    "中债新综合全价（总值）": "CBA00103.CS",

    "中债综合财富(总值)": "CBA00201.CS",
    "中债综合指数(总财富)": "CBA00201.CS",
    "中债综合指数收益率": "CBA00201.CS",

    "中债综合指数(全价)": "CBA00203.CS",
    "中债综合(全价)": "CBA00203.CS",
    "中债综合全价": "CBA00203.CS",
    "中债-综合全价（总值）": "CBA00203.CS",
    "中国债券综合全价": "CBA00203.CS",
    "中债总指数(全价)": "CBA00203.CS",
    "中债-总全价(总值)": "CBA00203.CS",

    "中债综合财富(1年以下)中债": "CBA00211.CS",
    "中债综合财富(1-3年)": "CBA00221.CS",
}

In [2]:
df = pd.read_csv('data/old_funds.csv')
df

,ts_code,name,found_date,invest_type,benchmark
0,002466.OF,博时裕新纯债A,20160330,债券型,中债综合指数(总财富)收益率*90%+1年期定期存款利率(税后)*10%
1,002411.OF,华夏新机遇A,20160330,灵活配置型,沪深300指数收益率*50%+上证国债指数收益率*50%
2,001864.OF,中海魅力长三角,20160330,灵活配置型,中证长三角龙头企业指数收益率*60%+中证全债指数收益率*40%
3,002552.OF,华夏恒利3个月定开,20160329,债券型,中债综合指数收益率
4,002489.OF,国泰民福策略价值A,20160329,灵活配置型,沪深300指数收益率*50%+中证综合债指数收益率*50%
...,...,...,...,...,...
2102,100016.OF,富国天源沪港深A,20020816,平衡型,沪深300指数收益率*65%+中债综合全价指数收益率*30%+同业存款利率*5%
2103,020001.OF,国泰金鹰增长,20020508,灵活配置型,沪深300指数收益率*80%+中证综合债指数收益率*20%
2104,000001.OF,华夏成长,20011218,成长型,NaN
2105,202001.OF,南方稳健成长,20010928,成长型,NaN


In [3]:
#   开筛
percent_pattern = re.compile(r'(\d+(?:\.\d+)?)\s*%')

def parse_benchmark(bmk: str):
    """
    输入原始 benchmark 字符串，输出:
        dict: {指数代码: 权重(float, 0-1)} 
    若含有任何不在 index_map 中的成分，返回 None
    """
    if not isinstance(bmk, str) or not bmk.strip():
        return None
    
    # 按 + / ＋ 拆分成各个成分
    parts = re.split(r'[+＋]', bmk)
    parts = [p.strip() for p in parts if p.strip()]
    
    code_weights = []   # 存 (code, weight or None)
    for part in parts:
        part_has_known_index = False
        part_codes = []

        for name, code in index_map.items():
            if name in part:
                part_has_known_index = True
                # 找该成分中的百分比
                m = percent_pattern.search(part)
                if m:
                    w = float(m.group(1)) / 100.0
                else:
                    w = None  # 后面统一处理
                part_codes.append((code, w))
        
        # 该成分中既没有任何已知指数名称，则直接丢掉整只基金
        if not part_has_known_index:
            return None
        
        # 理论上一个成分通常只对应一个指数；若匹配多个，就简单叠加
        code_weights.extend(part_codes)
    
    # 如果所有成分都匹配到了字典中的指数，但有权重缺失：
    weights = [w for _, w in code_weights]
    if any(w is None for w in weights):
        # 情况1：只有一个指数且没写百分比 -> 视为 100%
        if len(code_weights) == 1:
            code, _ = code_weights[0]
            return {code: 1.0}
        else:
            # 为避免瞎猜多指数权重，直接丢弃
            return None
    
    # 按代码聚合（防止同一指数出现在多段）
    agg = {}
    for code, w in code_weights:
        agg[code] = agg.get(code, 0.0) + w
    
    # 有些基准写法可能权重和略不为 1，这里简单归一化一下
    total_w = sum(agg.values())
    if total_w > 0:
        for c in agg:
            agg[c] = agg[c] / total_w
    
    return agg

In [4]:
parsed_dicts = []
parsed_strs = []

for bmk in df[BENCHMARK_COL]:
    res = parse_benchmark(bmk)
    parsed_dicts.append(res)
    if res is None:
        parsed_strs.append(None)
    else:
        # 转成 "代码*权重" 的形式，例如 "000300.SH*0.1+930930.CSI*0.9"
        s = "+".join(f"{code}*{weight:.4f}" for code, weight in res.items())
        parsed_strs.append(s)

df["parsed_benchmark_dict"] = parsed_dicts
df["parsed_benchmark_str"] = parsed_strs

# 只保留成功完全匹配到字典的基金
df_clean = df[df["parsed_benchmark_dict"].notna()].copy()

# df_clean.to_csv("funds_benchmark_clean.csv", index=False, encoding="utf-8-sig")
df_clean

,ts_code,name,found_date,invest_type,benchmark,parsed_benchmark_dict,parsed_benchmark_str
3,002552.OF,华夏恒利3个月定开,20160329,债券型,中债综合指数收益率,{'CBA00201.CS': 1.0},CBA00201.CS*1.0000
4,002489.OF,国泰民福策略价值A,20160329,灵活配置型,沪深300指数收益率*50%+中证综合债指数收益率*50%,"{'000300.SH': 0.5, 'H11009.CSI': 0.5}",000300.SH*0.5000+H11009.CSI*0.5000
6,002450.OF,平安睿享文娱A,20160329,灵活配置型,沪深300指数收益率*50%+中证综合债指数收益率*50%,"{'000300.SH': 0.5, 'H11009.CSI': 0.5}",000300.SH*0.5000+H11009.CSI*0.5000
7,002407.OF,前海开源恒远,20160328,灵活配置型,沪深300指数收益率*70%+中证全债指数收益率*30%,"{'000300.SH': 0.7, 'H11001.CSI': 0.3}",000300.SH*0.7000+H11001.CSI*0.3000
8,002067.OF,诺安精选回报A,20160328,灵活配置型,沪深300指数*60%+中证全债指数*40%,"{'000300.SH': 0.6, 'H11001.CSI': 0.4}",000300.SH*0.6000+H11001.CSI*0.4000
...,...,...,...,...,...,...,...
2091,121001.OF,国投瑞银融华债券,20030416,债券型,中债综合指数收益率*80%+沪深300指数收益率*20%,"{'CBA00201.CS': 0.8, '000300.SH': 0.2}",CBA00201.CS*0.8000+000300.SH*0.2000
2095,001001.OF,华夏债券AB,20021023,债券型,中证综合债券指数,{'H11009.CSI': 1.0},H11009.CSI*1.0000
2097,213001.OF,宝盈鸿利收益A,20021008,灵活配置型,沪深300指数收益率*65%+中债综合指数收益率(全价)*35%,"{'000300.SH': 0.65, 'CBA00201.CS': 0.35}",000300.SH*0.6500+CBA00201.CS*0.3500
2100,161601.OF,融通新蓝筹,20020913,平衡型,沪深300指数收益率*75%+中债综合全价(总值)指数收益率*25%,"{'000300.SH': 0.75, 'CBA00203.CS': 0.25}",000300.SH*0.7500+CBA00203.CS*0.2500


In [5]:
df_clean.to_csv(f"data/sample_funds.csv")